```python
# 缩点模板 https://www.luogu.com.cn/record/159334791

# 返回每个scc中点的集合
def find_SCC(graph):
    SCC, S, P = [], [], []
    depth = [0] * len(graph)
 
    stack = list(range(len(graph)))
    while stack:
        node = stack.pop()
        if node < 0:
            d = depth[~node] - 1
            if P[-1] > d:
                SCC.append(S[d:])
                del S[d:], P[-1]
                for node in SCC[-1]:
                    depth[node] = -1
        elif depth[node] > 0:
            while P[-1] > depth[node]:
                P.pop()
        elif depth[node] == 0:
            S.append(node)
            P.append(len(S))
            depth[node] = len(S)
            stack.append(~node)
            stack += graph[node]
    return SCC[::-1]

def get_newg(scc):
    ID = [0] * n
    scc_cnt = 0
    scc_w = [0] * n
    for v in scc:
        for i in v:
            ID[i] = scc_cnt
            scc_w[scc_cnt] += w[i]
        scc_cnt += 1
        
    new_g = [[] for _ in range(n)]
    left = [0] * n
    for i in range(n):
        for j in g[i]:
            a, b = ID[i], ID[j]
            if a == b:
                continue
            new_g[a].append(b)
            left[b] += 1
    return new_g, left, scc_w


n, m = MII()
g = [[] for _ in range(n)]
w = LII()
for _ in range(m):
    x, y = GMI()
    g[x].append(y)

scc = find_SCC(g)

ID = [0] * n
scc_cnt = 0
for v in scc:
    for i in v:
        ID[i] = scc_cnt
    scc_cnt += 1


""""
inf = float('inf')
n, m = MII()
N, M = int(1e4 + 10), int(5e4 + 10)
g = [[] for _ in range(N + 1)]

dfn = [0] * N  # 被访问到的实际时间点
low = [0] * N  # 能回到的最高点
id = [0] * N
timestamp = 0
stk = []
in_stk = [False] * N
scc_cnt = top = 0
Size = [0] * N
doubt = [0] * N

def tarjan(u:int):
    global timestamp, scc_cnt
    dfn[u] = low[u] = timestamp = timestamp + 1   # 时间戳，默认相等
    stk.append(u)   # 入栈
    in_stk[u] = True
    for x in g[u]:
        if not dfn[x]:  # 未访问过该结点
            tarjan(x)
            low[u] = min(low[u], low[x])   # 有可能通过该点回到过去
        elif in_stk[x]:
            low[u] = min(low[u], dfn[x])   # 在栈中我们认为栈下面的点时间戳一定是小于当前点的
    if dfn[u] == low[u]:  # 回到了自己
        scc_cnt += 1
        while True:
            y = stk.pop()  # 节点全部出栈
            in_stk[y] = False
            id[y] = scc_cnt  # 结点属于哪个强连通分量
            Size[scc_cnt] += 1  # 连通分量的大小+1
            if y == u:  # 出栈结束
                break
                
for _ in range(m):
    x, y = MII()
    g[x].append(y)

for i in range(1, n + 1):
    if not dfn[i]:
        tarjan(i)

for i in range(1, n + 1):
    for j in g[i]:
        a, b = id[i], id[j]  # scc内部互相可达
        if a != b:
            doubt[a] += 1
            
zeros = sum = 0
for i in range(1, scc_cnt + 1):
    if not doubt[i]:
        zeros += 1
        sum += Size[i]
        if zeros > 1:
            sum = 0
            break
print(sum)


"""
```

```python3
# 任意交换两个结点需要的最少操作数
https://leetcode.cn/problems/minimum-number-of-operations-to-sort-a-binary-tree-by-level/description/
def min_swaps(arr):
    n = len(arr)
    arrPos = [*enumerate(arr)]
    arrPos.sort(key = lambda it:it[1])
    vis = {i:False for i in range(n)}
    ans = 0
    for i in range(n):
        if vis[i] or arrPos[i][0] == i:
            continue
        cycle_size = 0
        j = i
        while not vis[j]:
            vis[j] = True
            j = arrPos[j][0]
            cycle_size += 1
        if cycle_size > 0:
            ans += (cycle_size - 1)
    return ans
```

```python3
# https://codeforces.com/contest/1974/problem/F
# 二维数点
# 直接传入原数组,查询的时候不用给下标减一[1, n]
class FenwickTree:
    def __init__(self, x):
        """transform list into BIT"""
        self.bit = x
        for i in range(len(x)):
            j = i | (i + 1)
            if j < len(x):
                x[j] += x[i]

    def update(self, idx, x):
        """updates bit[idx] += x"""
        while idx < len(self.bit):
            self.bit[idx] += x
            idx |= idx + 1

    def query(self, end):
        """calc sum(bit[:end])"""
        x = 0
        while end:
            x += self.bit[end - 1]
            end &= end - 1
        return x

from bisect import bisect_left
# x1, y1为左下角坐标，x2, y2为右上角坐标。查询矩形内的点的个数（包括边界）
def solve(pts, qrs):
    n, m = len(pts), len(qrs)
    # 离散化坐标y
    Y = set()
    for x, y in pts:
        Y.add(y)
    for x1, y1, x2, y2 in qrs:
        Y.add(y1)
        Y.add(y2)
    Y = sorted(Y)

    op = []
    for i, (x, y) in enumerate(pts):
        y = bisect_left(Y, y) + 1
        op.append((x, y, i, 0))    # 加点操作
    
    # 矩形面积计算原理：s[x2][y2] + s[x1 - 1][y1 - 1] - s[x1 - 1][y2] - s[x2][y1 - 1]
    for i, (x1, y1, x2, y2) in enumerate(qrs):
        y1 = bisect_left(Y, y1) + 1
        y2 = bisect_left(Y, y2) + 1
        op.append((x2, y2, i, 1))
        op.append((x1 - 1, y1 - 1, i, 1))
        op.append((x1 - 1, y2, i, 2))
        op.append((x2, y1 - 1, i, 2))

    op.sort(key=lambda x:x[0])  # 按照x坐标排序，其余相对位置不变

    bit = FenwickTree([0] * (len(Y) + 10))
    res = [0] * m
    for i, (x, y, idx, t) in enumerate(op):
        if t == 0:
            bit.update(y, 1)
        elif t == 1:
            res[idx] += bit.query(y + 1)
        else:
            res[idx] -= bit.query(y + 1)
    return res
```

```python3
class PrimeTable:
    def __init__(self, n:int) -> None:
        self.n = n
        self.primes = []
        self.max_div = list(range(n+1))
        self.max_div[1] = 1
        self.phi = list(range(n+1))
 
        for i in range(2, n + 1):
            if self.max_div[i] == i:
                self.primes.append(i)
                for j in range(i, n+1, i):
                    self.max_div[j] = i
                    self.phi[j] = self.phi[j] // i * (i-1)
 
    def is_prime(self, x:int):
        if x < 2: return False
        if x <= self.n: return self.max_div[x] == x
        for p in self.primes:
            if p * p > x: break
            if x % p == 0: return False
        return True
 
    def prime_factorization(self, x:int):
        if x > self.n:
            for p in self.primes:
                if p * p > x: break
                if x <= self.n: break
                if x % p == 0:
                    cnt = 0
                    while x % p == 0: cnt += 1; x //= p
                    yield p, cnt
        while (1 < x and x <= self.n):
            p, cnt = self.max_div[x], 0
            while x % p == 0: cnt += 1; x //= p
            yield p, cnt
        if x >= self.n and x > 1:
            yield x, 1
 
    def get_factors(self, x:int):
        factors = [1]
        for p, b in self.prime_factorization(x):
            n = len(factors)
            for j in range(1, b+1):
                for d in factors[:n]:
                    factors.append(d * (p ** j))
        return factors
```

```python3
def isPrimeMR(n):
    if n <= 1:
        return 0
    if n == 2 or n == 7 or n == 61:
        return 1
    d = n - 1
    d = d // (d & -d)
    L = [2, 7, 61] if n < 1 << 32 else [2, 3, 5, 7, 11, 13, 17] if n < 1 << 48 else [2, 3, 5, 7, 11, 13, 17, 19, 23, 29,
                                                                                     31, 37]
    for a in L:
        t = d
        y = pow(a, t, n)
        if y == 1: continue
        while y != n - 1:
            y = y * y % n
            if y == 1 or t == n - 1: return 0
            t <<= 1
    return 1

def findFactorRho(n):
    m = 1 << n.bit_length() // 8
    for c in range(1, 99):
        f = lambda x: (x * x + c) % n
        y, r, q, g = 2, 1, 1, 1
        while g == 1:
            x = y
            for i in range(r):
                y = f(y)
            k = 0
            while k < r and g == 1:
                ys = y
                for i in range(min(m, r - k)):
                    y = f(y)
                    q = q * abs(x - y) % n
                g = gcd(q, n)
                k += m
            r <<= 1
        if g == n:
            g = 1
            while g == 1:
                ys = f(ys)
                g = gcd(abs(x - ys), n)
        if g < n:
            if isPrimeMR(g):
                return g
            elif isPrimeMR(n // g):
                return n // g
            return findFactorRho(g)
        
def primeFactor(n):
    i = 2
    ret = {}
    rhoFlg = 0
    while i * i <= n:
        k = 0
        while n % i == 0:
            n //= i
            k += 1
        if k: ret[i] = k
        i += i % 2 + (3 if i % 3 == 1 else 1)
        if i == 101 and n >= 2 ** 20:
            while n > 1:
                if isPrimeMR(n):
                    ret[n], n = 1, 1
                else:
                    rhoFlg = 1
                    j = findFactorRho(n)
                    k = 0
                    while n % j == 0:
                        n //= j
                        k += 1
                    ret[j] = k
    if n > 1: ret[n] = 1
    if rhoFlg: ret = {x: ret[x] for x in sorted(ret)}
    return ret

def divisors(N):
    pf = primeFactor(N)
    ret = [1]
    for p in pf:
        ret_prev = ret
        ret = []
        for i in range(pf[p] + 1):
            for r in ret_prev:
                ret.append(r * (p ** i))
    return sorted(ret)
```